# Phase 0 — SPY Chain Data Sanity Check

Goal: verify `pilot_data/spy_chains.csv.gz` is usable **before** building any strategy on top of it.

The chain is accepted as usable only if all of the following hold:

1. Schema matches expectations, dtypes are reasonable, row count is non-trivial.
2. Every NYSE trading day in the chain's date range has a chain (no silent gaps > a few days).
3. For a representative day (2023-06-15): strikes are dense around spot, bid/ask/last are populated, expiries include at least one maturity under 45 days.
4. The manually-computed ATM 30-day IV for 2023-06-15 is in the 10–30% range.
5. The full ATM-30d-IV time series tracks VIX with Pearson correlation ≥ 0.90.

If any check fails, **stop** and fix the root cause before anything downstream.

In [ ]:
import sys, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('..').resolve()
CHAIN_PATH = PROJECT_ROOT / 'option_volatility_crush.ipynb' / 'pilot_data' / 'spy_chains.csv.gz'

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

assert CHAIN_PATH.exists(), f'Chain file not found at {CHAIN_PATH}'
print('Chain file:', CHAIN_PATH)
print('Size:       {:.1f} MB'.format(CHAIN_PATH.stat().st_size / 1e6))

SANITY = {}

## 1. Load and inspect schema

In [ ]:
chain = pd.read_csv(
    CHAIN_PATH,
    parse_dates=['date', 'expiration', 'fetch_date'],
)

# Alpha Vantage writes '-' for missing numerics, which silently drags the
# whole column to object dtype. Coerce sentinels to NaN so numerics are real.
num_cols = ['strike','last','mark','bid','ask','volume','open_interest',
            'implied_volatility','delta','gamma','theta','vega','rho',
            'bid_size','ask_size']
orig_iv_null = chain['implied_volatility'].isna().sum()
for c in num_cols:
    chain[c] = pd.to_numeric(chain[c], errors='coerce')
new_iv_null = chain['implied_volatility'].isna().sum()
coerced_iv = int(new_iv_null - orig_iv_null)
print(f'Coerced {coerced_iv:,} non-numeric implied_volatility sentinels to NaN')
if coerced_iv:
    outage_days = (chain.loc[chain['implied_volatility'].isna(), 'date']
                        .dt.date.value_counts().sort_values(ascending=False).head(5))
    print('Top 5 dates by missing-IV row count:')
    print(outage_days.to_string())
    print()

print('Rows     :', f'{len(chain):,}')
print('Columns  :', list(chain.columns))
print('Date min :', chain['date'].min().date())
print('Date max :', chain['date'].max().date())
print('Trade days in file :', chain['date'].nunique())
print()
print('Dtypes:')
print(chain.dtypes)
print()
print('Null fraction per column:')
print((chain.isna().mean() * 100).round(2).astype(str) + ' %')

SANITY['rows']       = len(chain)
SANITY['date_min']   = chain['date'].min().date()
SANITY['date_max']   = chain['date'].max().date()
SANITY['n_days']     = chain['date'].nunique()
SANITY['iv_coerced'] = coerced_iv
SANITY['pass_load']  = len(chain) > 1_000_000 and chain['date'].nunique() > 500


## 2. Date coverage vs. NYSE trading calendar

We only expect coverage from the earliest date in the chain onward — not from 2021-01-01. Compare unique chain dates to the NYSE trading-day calendar. Any missing trading day is a gap.

In [ ]:
start = chain['date'].min()
end   = chain['date'].max()

# Prefer pandas_market_calendars; fall back to US federal holiday calendar.
try:
    import pandas_market_calendars as mcal
    nyse = mcal.get_calendar('NYSE')
    trading_days = nyse.valid_days(start_date=start, end_date=end).tz_localize(None).normalize()
    cal_source = 'pandas_market_calendars (NYSE)'
except ImportError:
    from pandas.tseries.holiday import USFederalHolidayCalendar
    from pandas.tseries.offsets import CustomBusinessDay
    bd = CustomBusinessDay(calendar=USFederalHolidayCalendar())
    trading_days = pd.date_range(start, end, freq=bd).normalize()
    cal_source = 'USFederalHolidayCalendar (approximate — install pandas_market_calendars for exact NYSE)'

chain_days = pd.DatetimeIndex(sorted(chain['date'].dt.normalize().unique()))
missing = trading_days.difference(chain_days)
extra   = chain_days.difference(trading_days)

print('Calendar source   :', cal_source)
print('Expected days     :', len(trading_days))
print('Chain days        :', len(chain_days))
print('Missing (gaps)    :', len(missing))
print('Extra (non-NYSE)  :', len(extra))

if len(missing):
    print('\nFirst 15 missing:', [d.date().isoformat() for d in missing[:15]])
    print('Last 15 missing :', [d.date().isoformat() for d in missing[-15:]])

coverage = 1 - len(missing) / max(len(trading_days), 1)
SANITY['coverage']      = coverage
SANITY['n_missing']     = len(missing)
SANITY['pass_coverage'] = coverage >= 0.98
print(f'\nCoverage: {coverage:.2%}  ->  {"PASS" if SANITY["pass_coverage"] else "FAIL"}')

## 3. Single representative day: 2023-06-15

Confirm strikes are dense around spot, bid/ask/last aren't all NaN, and at least one expiry is < 45 days out.

In [ ]:
SAMPLE_DATE = pd.Timestamp('2023-06-15')
day = chain[chain['date'] == SAMPLE_DATE].copy()

if day.empty:
    nearest = chain['date'].iloc[(chain['date'] - SAMPLE_DATE).abs().argsort()].iloc[0]
    print(f'No rows on {SAMPLE_DATE.date()}. Nearest available: {nearest.date()}')
    SAMPLE_DATE = nearest
    day = chain[chain['date'] == SAMPLE_DATE].copy()

day['dte'] = (day['expiration'] - day['date']).dt.days

print(f'Date              : {SAMPLE_DATE.date()}')
print(f'Contracts         : {len(day):,}')
print(f'Calls / Puts      : {(day["type"]=="call").sum():,} / {(day["type"]=="put").sum():,}')
print(f'Strike min / max  : {day["strike"].min():.2f} / {day["strike"].max():.2f}')
print(f'Unique strikes    : {day["strike"].nunique()}')
print(f'Unique expirations: {day["expiration"].nunique()}')
print(f'DTE min / max     : {day["dte"].min()} / {day["dte"].max()}')
print()
print('DTE distribution (unique expiries):')
print(day[['expiration','dte']].drop_duplicates().sort_values('dte').head(15).to_string(index=False))
print()
print('Quote availability (non-NaN fraction):')
for col in ['bid','ask','mark','last','implied_volatility']:
    print(f'  {col:<20s} {(1 - day[col].isna().mean()):.2%}')

has_near_term = (day['dte'] < 45).any()
has_quotes    = day[['bid','ask','last']].notna().any(axis=1).mean() > 0.5
SANITY['pass_sample_day'] = bool(has_near_term and has_quotes and len(day) > 100)
print(f'\nNear-term expiry (<45d): {has_near_term}')
print(f'Quote coverage >50%    : {has_quotes}')
print(f'Sample day check       : {"PASS" if SANITY["pass_sample_day"] else "FAIL"}')

## 4. Manual ATM 30-day IV for the sample day

Procedure:
- Infer spot from put-call parity: for the nearest non-zero-DTE expiration, the strike where `|call_mid - put_mid - (S - K*e^{-rT})|` is minimized gives an implied forward; we approximate `S ≈ K*` where `|call - put|` is smallest on that expiration.
- Pick the two expirations straddling 30 DTE.
- On each, take the call and put whose strikes bracket spot, average their IVs at the ATM strike (closest to spot).
- Linearly interpolate the two ATM IVs in calendar time to exactly 30 DTE (VIX-style).
- Expect result in ~10–30% for SPY in normal conditions.

In [ ]:
def infer_spot(day_df):
    """Use put-call parity on the nearest-dated expiry: spot ≈ strike where |call_mark - put_mark| is minimized."""
    dfe = day_df[day_df['dte'] > 0].copy()
    if dfe.empty:
        return np.nan
    near_exp = dfe['expiration'].value_counts().idxmax()
    sub = dfe[dfe['expiration'] == near_exp]
    wide = sub.pivot_table(index='strike', columns='type', values='mark', aggfunc='first')
    wide = wide.dropna(subset=['call','put'])
    if wide.empty:
        return np.nan
    diff = (wide['call'] - wide['put']).abs()
    return float(diff.idxmin())

def atm_iv_for_expiry(day_df, expiration, spot):
    sub = day_df[(day_df['expiration'] == expiration) & day_df['implied_volatility'].notna()].copy()
    if sub.empty or not np.isfinite(spot):
        return np.nan
    atm_strike = sub.iloc[(sub['strike'] - spot).abs().argsort()]['strike'].iloc[0]
    legs = sub[sub['strike'] == atm_strike]
    return float(legs['implied_volatility'].mean())

def interp_30d_iv(day_df):
    dfe = day_df[day_df['dte'] > 0].drop_duplicates('expiration').sort_values('dte')
    if dfe.empty:
        return np.nan, {}
    spot = infer_spot(day_df)
    exps = dfe[['expiration','dte']].drop_duplicates().sort_values('dte')
    below = exps[exps['dte'] <= 30].tail(1)
    above = exps[exps['dte'] >= 30].head(1)
    details = {'spot': spot}
    if below.empty or above.empty:
        anchor = exps.iloc[(exps['dte'] - 30).abs().argsort()].iloc[0]
        iv = atm_iv_for_expiry(day_df, anchor['expiration'], spot)
        details.update(exp_lo=anchor['expiration'], exp_hi=anchor['expiration'],
                       dte_lo=int(anchor['dte']), dte_hi=int(anchor['dte']),
                       iv_lo=iv, iv_hi=iv)
        return iv, details
    lo, hi = below.iloc[0], above.iloc[0]
    iv_lo = atm_iv_for_expiry(day_df, lo['expiration'], spot)
    iv_hi = atm_iv_for_expiry(day_df, hi['expiration'], spot)
    if lo['dte'] == hi['dte']:
        iv = np.nanmean([iv_lo, iv_hi])
    else:
        w = (30 - lo['dte']) / (hi['dte'] - lo['dte'])
        iv = (1 - w) * iv_lo + w * iv_hi
    details.update(exp_lo=lo['expiration'], exp_hi=hi['expiration'],
                   dte_lo=int(lo['dte']), dte_hi=int(hi['dte']),
                   iv_lo=iv_lo, iv_hi=iv_hi)
    return iv, details

iv_30d, details = interp_30d_iv(day)
print(f'Spot (parity)      : {details["spot"]:.2f}')
print(f'Bracket expiries   : {details["exp_lo"].date()} ({details["dte_lo"]}d)  ->  {details["exp_hi"].date()} ({details["dte_hi"]}d)')
print(f'ATM IV at each     : {details["iv_lo"]:.4f}  /  {details["iv_hi"]:.4f}')
print(f'Interpolated 30d IV: {iv_30d:.4f}  ({iv_30d*100:.2f}%)')

SANITY['sample_iv_30d']    = iv_30d
SANITY['pass_sample_iv']   = bool(np.isfinite(iv_30d) and 0.05 <= iv_30d <= 0.40)
print(f'Sample-day IV check: {"PASS" if SANITY["pass_sample_iv"] else "FAIL"} (expected 10–30%)')

## 5. ATM 30-day IV time series vs. VIX

Apply the same ATM-30d-IV computation to every trading day. Overlay VIX (from `market_vol_context.fetch_vix_data`). They should track each other with correlation ≥ 0.90 — VIX has a ~30-day horizon by construction and SPY ATM IV is the underlying it's built from.

This cell is the most expensive step (one computation per trading day). Expect a few minutes.

In [ ]:
chain['dte'] = (chain['expiration'] - chain['date']).dt.days
only_fwd = chain[chain['dte'] > 0]
grouped = only_fwd.groupby('date', sort=True)

records = []
for i, (d, g) in enumerate(grouped):
    iv, det = interp_30d_iv(g)
    records.append({'date': d, 'atm_iv_30d': iv, 'spot': det.get('spot', np.nan)})
    if (i + 1) % 100 == 0:
        print(f'  processed {i+1}/{grouped.ngroups} days')

ts = pd.DataFrame(records).set_index('date').sort_index()
print(f'\nTime series rows  : {len(ts)}')
print(f'NaN ATM IV days   : {ts["atm_iv_30d"].isna().sum()}')
print(ts['atm_iv_30d'].describe())

In [ ]:
from market_vol_context import fetch_vix_data

vix = fetch_vix_data(
    start_date=ts.index.min().strftime('%Y-%m-%d'),
    end_date=(ts.index.max() + pd.Timedelta(days=1)).strftime('%Y-%m-%d'),
)
vix.index = pd.to_datetime(vix.index).normalize()

joined = ts.join(vix[['vix_close']], how='inner')
joined['atm_iv_30d_pct'] = joined['atm_iv_30d'] * 100

corr = joined[['atm_iv_30d_pct','vix_close']].corr().iloc[0,1]
print(f'Overlapping days  : {len(joined)}')
print(f'Correlation (ATM IV pct vs VIX close): {corr:.4f}')

SANITY['corr_vix']        = float(corr)
SANITY['pass_vix_corr']   = bool(np.isfinite(corr) and corr >= 0.90)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(joined.index, joined['atm_iv_30d_pct'], label='SPY ATM 30d IV (from chain)', lw=1.2)
ax.plot(joined.index, joined['vix_close'], label='VIX (Yahoo ^VIX close)', lw=1.2, alpha=0.8)
ax.set_title(f'SPY ATM 30-day IV vs. VIX   (corr = {corr:.3f})')
ax.set_ylabel('Implied volatility, %')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(joined['vix_close'], joined['atm_iv_30d_pct'], s=6, alpha=0.5)
lim_lo = float(min(joined['vix_close'].min(), joined['atm_iv_30d_pct'].min())) - 1
lim_hi = float(max(joined['vix_close'].max(), joined['atm_iv_30d_pct'].max())) + 1
ax.plot([lim_lo, lim_hi], [lim_lo, lim_hi], ls='--', color='k', lw=0.8, label='y = x')
ax.set_xlabel('VIX close')
ax.set_ylabel('SPY ATM 30d IV (%)')
ax.set_title('Chain-derived ATM IV vs. VIX')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

In [ ]:
checks = [
    ('Load / schema',              SANITY.get('pass_load')),
    ('Date coverage (>=98%)',      SANITY.get('pass_coverage')),
    ('Sample day 2023-06-15 OK',   SANITY.get('pass_sample_day')),
    ('Sample ATM 30d IV in 5–40%', SANITY.get('pass_sample_iv')),
    ('VIX correlation >= 0.90',    SANITY.get('pass_vix_corr')),
]
for name, ok in checks:
    print(f'  [{"PASS" if ok else "FAIL"}] {name}')
print()
print('Details:')
for k, v in SANITY.items():
    print(f'  {k:<20s} {v}')

all_pass = all(ok for _, ok in checks)
print(f'\nOverall: {"ALL CHECKS PASSED — safe to proceed" if all_pass else "FAIL — investigate before building on this data"}')